# Playground — Step 4: Family 2 Score-Space Aggregators + Normalization

New module `frames.representations.aggregators`:

| function | semantics |
|---|---|
| `weighted_sum(s, w)` | F2.a — OR-like; one strong concept compensates; negative weights repel |
| `softmin(s, w, tau)` | F2.b — AND-like; a candidate is only as good as its worst concept; τ→0 = hard min |
| `constrained(s, w, thresholds)` | F2.c — maximize concept 0 subject to the rest ≥ thresholds |
| `normalize_scores(s, method, dim)` | per-step per-concept z-score / rank normalization over the candidate pool |

All plug into `_generate_guided` / beam via the `scorer` argument; normalization via the
new `normalize="zscore"` flag (also on `generate_with_topk_multi_guide`).

**Why normalization is mandatory:** frame correlation is linear in the concept frame, so an
unnormalized weighted sum of per-concept scores ≡ scoring against one averaged frame —
i.e. Family 2 without normalization is just Family 1 in disguise. Section 2 shows this live.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # repo root, so `frames` imports work

In [ ]:
import torch

from frames.representations import FrameUnembeddingRepresentation, aggregators

In [ ]:
fur = FrameUnembeddingRepresentation.from_model_id(
    "hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4",
    device_map="cuda:0",
    torch_dtype=torch.float16,
)

MIN_LEMMAS = 11
MAX_TOKENS = 3
K = 3
STEPS = 16


def chat(user: str, assistant: str = "") -> str:
    return (
        f"<|start_header_id|>user<|end_header_id|>{user}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>{assistant}"
    )


def answer(text: str) -> str:
    return text.split("assistant<|end_header_id|>")[-1]


PROMPTS = [
    chat("Tell me a short story.", "Once"),
    chat("Name things that make people happy.", "1."),
]

concept_joy = fur.get_concept_cached("joy.n.01", MIN_LEMMAS, MAX_TOKENS)
concept_dog = fur.get_concept_cached("dog.n.01", MIN_LEMMAS, MAX_TOKENS)
CONCEPTS = [concept_joy, concept_dog]

## 1 — Aggregator intuition on toy scores (no model needed)

Three candidates × two concepts, concept 2 with an inflated score scale:

In [ ]:
toy = torch.tensor(
    [
        [1.0, 0.0],  # A: concept 1 only
        [0.0, 10.0],  # B: concept 2 only (inflated scale)
        [0.9, 9.0],  # C: near-best on both
    ]
)
w = torch.tensor([0.5, 0.5])
z = aggregators.normalize_scores(toy, "zscore", dim=0)

print("raw weighted_sum:    ", aggregators.weighted_sum(toy, w), "-> picks B (scale wins)")
print("zscored weighted_sum:", aggregators.weighted_sum(z, w).round(decimals=3), "-> picks C (balance wins)")
print("zscored softmin:     ", aggregators.softmin(z, w, tau=0.1).round(decimals=3), "-> AND semantics")
print("constrained:         ", aggregators.constrained(toy, w, thresholds=[5.0]), "-> A violates, others pass")

## 2 — The linearity confound, live

Mean-frame steering (F1.a) vs **unnormalized** weighted sum (F2.a-raw) vs **z-scored**
weighted sum (true F2.a). Expect raw F2.a to behave like F1.a; z-scoring should diverge.

In [ ]:
texts_mean, _ = fur.quick_generate_with_topk_subspace_guide(
    PROMPTS, guides=[["joy.n.01"], ["dog.n.01"]],
    min_lemmas_per_synset=MIN_LEMMAS, max_token_count=MAX_TOKENS, k=K, steps=STEPS,
)
texts_raw, _ = fur.generate_with_topk_multi_guide(
    PROMPTS, guides=CONCEPTS, k=K, steps=STEPS,
)
texts_z, _ = fur.generate_with_topk_multi_guide(
    PROMPTS, guides=CONCEPTS, k=K, steps=STEPS, normalize="zscore",
)

for mean_t, raw_t, z_t in zip(texts_mean, texts_raw, texts_z):
    print("F1.a mean-frame:", answer(mean_t)[:110])
    print("F2.a raw:       ", answer(raw_t)[:110])
    print("F2.a zscore:    ", answer(z_t)[:110])
    print()

## 3 — OR vs AND: weighted sum vs softmin

Same two concepts, z-scored. `weighted_sum` lets one concept dominate a step;
`softmin` demands every step satisfy both. Try different `tau` values.

In [ ]:
texts_and, probe_and = fur.generate_with_topk_multi_guide(
    PROMPTS, guides=CONCEPTS, k=K, steps=STEPS,
    normalize="zscore", scorer=aggregators.softmin_scorer(tau=0.5),
)

for or_t, and_t in zip(texts_z, texts_and):
    print("OR  (sum):    ", answer(or_t)[:110])
    print("AND (softmin):", answer(and_t)[:110])
    print()

## 4 — Constrained: steer to joy, subject to dog-alignment above threshold

Concept 0 is maximized; the rest must clear their thresholds (on z-scored scores,
0.0 = pool average). Uses the finite-penalty factory (safe under cumulative summing).

In [ ]:
texts_con, _ = fur.generate_with_topk_multi_guide(
    PROMPTS, guides=CONCEPTS, k=K, steps=STEPS,
    normalize="zscore",
    scorer=aggregators.constrained_scorer(thresholds=[0.0]),
)
for t in texts_con:
    print(answer(t)[:140], "\n")

## 5 — Negative steering (signed weights)

Attract joy, repel dog — generalizes the paper's bias-remediation to score space.
Compare with differential frame guidance (`joy - dog`) which does this in frame space.

In [ ]:
texts_neg, _ = fur.generate_with_topk_multi_guide(
    PROMPTS, guides=CONCEPTS, weights=[1.0, -1.0], k=K, steps=STEPS,
    normalize="zscore",
)
texts_diff, _ = fur.quick_generate_with_topk_guide(
    PROMPTS, guide=["joy.n.01", "dog.n.01"],
    min_lemmas_per_synset=MIN_LEMMAS, max_token_count=MAX_TOKENS, k=K, steps=STEPS,
)

for neg_t, diff_t in zip(texts_neg, texts_diff):
    print("score-space (w=[1,-1]): ", answer(neg_t)[:110])
    print("frame-space (joy - dog):", answer(diff_t)[:110])
    print()

## 6 — Everything also works under beam search

The beam variant shares the scorer/normalize interface.

In [ ]:
texts_beam_and, _ = fur.generate_with_topk_beam_guide(
    PROMPTS, concepts=CONCEPTS, k=K, steps=STEPS,
    normalize="zscore", scorer=aggregators.softmin_scorer(tau=0.5),
)
for t in texts_beam_and:
    print(answer(t)[:140], "\n")